In [0]:
/**************************************************************************\
* CLEAR THE PRIMARY WORKTABLES PRIOR TO LOADING                            *
\**************************************************************************/

-- Converted Teradata database-qualified name to the current Databricks catalog/schema.
DELETE FROM AML_KYCSYS_RAW;


/**************************************************************************\
* BEGIN MULTILOAD PROCESSING                                               *
* ------------------------------------------------------------------------ *
* KYCSYS customer due-diligence delta extract (fixed width, 500 bytes)     *
* PY = party, AC = account, PA = party/account role, AL = alias            *
* Header (HD) and trailer (TR) records are filtered by the APPLY           *
\**************************************************************************/

-- TODO: manual review required — MultiLoad → COPY INTO staging + MERGE INTO (confirm source path).
-- Converted the MultiLoad fixed-width layout to a one-column Databricks staging table.
CREATE OR REPLACE TEMP TABLE staging_kycsys_raw (
    raw_line STRING
);

-- Converted the MultiLoad text import to COPY INTO; the relative source path is retained pending confirmation.
COPY INTO staging_kycsys_raw
FROM '../data/kycref.kycsys.dat'
FILEFORMAT = TEXT;

-- Converted the fixed-width fields, header/trailer APPLY filter, and insert-only DML to MERGE.
MERGE INTO AML_KYCSYS_RAW AS target
USING (
    SELECT
        SUBSTRING(raw_line, 1, 2) AS a,
        SUBSTRING(raw_line, 3, 26) AS b,
        SUBSTRING(raw_line, 29, 472) AS c
    FROM staging_kycsys_raw
    WHERE SUBSTRING(raw_line, 1, 2) NOT IN ('HD', 'TR')
) AS source
ON FALSE
WHEN NOT MATCHED THEN
    INSERT (a, b, c)
    VALUES (source.a, source.b, source.c);

-- Removed the MultiLoad work/error tables and session controls; drop the staging table after loading.
DROP TABLE staging_kycsys_raw;
